In [1]:
# ================================
# 1. SETUP
# ================================

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import pandas as pd
import json
import time

options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

# Página base (clave para evitar bloqueo)
driver.get("https://resultadoelectoral.onpe.gob.pe/main/presidenciales")
time.sleep(5)

# ================================
# 2. REGIONES
# ================================

departamentos = [
    ("010000","AMAZONAS"),
    ("020000","ANCASH"),
    ("030000","APURIMAC"),
    ("040000","AREQUIPA"),
    ("050000","AYACUCHO"),
    ("060000","CAJAMARCA"),
    ("240000","CALLAO"),
    ("070000","CUSCO"),
    ("080000","HUANCAVELICA"),
    ("090000","HUANUCO"),
    ("100000","ICA"),
    ("110000","JUNIN"),
    ("120000","LA LIBERTAD"),
    ("130000","LAMBAYEQUE"),
    ("140000","LIMA"),
    ("150000","LORETO"),
    ("160000","MADRE DE DIOS"),
    ("170000","MOQUEGUA"),
    ("180000","PASCO"),
    ("190000","PIURA"),
    ("200000","PUNO"),
    ("210000","SAN MARTIN"),
    ("220000","TACNA"),
    ("230000","TUMBES"),
    ("250000","UCAYALI")
]

# ================================
# 3. EXTRAER PROVINCIAS
# ================================

provincias = []

for ubigeo_dep, nombre_dep in departamentos:
    
    url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/ubigeos/provincias?idEleccion=10&idAmbitoGeografico=1&idUbigeoDepartamento={ubigeo_dep}"
    
    print(f"Provincias {nombre_dep}...")

    driver.get(url)
    time.sleep(2)

    texto = driver.find_element("tag name", "body").text

    try:
        data = json.loads(texto)["data"]

        for p in data:
            provincias.append({
                "departamento": nombre_dep,
                "ubigeo_dep": ubigeo_dep,
                "ubigeo_prov": p["ubigeo"],
                "provincia": p["nombre"]
            })

    except Exception as e:
        print(f"❌ Error provincias {nombre_dep}: {e}")

df_provincias = pd.DataFrame(provincias)

print("✅ Provincias cargadas")

# ================================
# 4. VOTOS POR PROVINCIA
# ================================

votos = []

for i, row in df_provincias.iterrows():

    url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/eleccion-presidencial/participantes-ubicacion-geografica-nombre?tipoFiltro=ubigeo_nivel_02&idAmbitoGeografico=1&ubigeoNivel1={row['ubigeo_dep']}&ubigeoNivel2={row['ubigeo_prov']}&idEleccion=10"

    print(f"Votos {row['provincia']} ({row['departamento']})")

    driver.get(url)
    time.sleep(2)

    texto = driver.find_element("tag name", "body").text

    try:
        data = json.loads(texto)["data"]

        for fila in data:
            votos.append({
                "departamento": row["departamento"],
                "provincia": row["provincia"],
                "candidato": fila["nombreCandidato"],
                "partido": fila["nombreAgrupacionPolitica"],
                "votos": fila.get("totalVotosValidos", 0)
            })

    except Exception as e:
        print(f"❌ Error votos {row['provincia']}: {e}")

df_votos = pd.DataFrame(votos)

print("✅ Votos cargados")

# ================================
# 5. ACTAS POR PROVINCIA
# ================================

actas = []

for i, row in df_provincias.iterrows():

    url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/resumen-general/totales?idAmbitoGeografico=1&idEleccion=10&tipoFiltro=ubigeo_nivel_02&idUbigeoDepartamento={row['ubigeo_dep']}&idUbigeoProvincia={row['ubigeo_prov']}"

    print(f"Actas {row['provincia']} ({row['departamento']})")

    driver.get(url)
    time.sleep(2)

    texto = driver.find_element("tag name", "body").text

    try:
        data = json.loads(texto)["data"]

        actas.append({
            "departamento": row["departamento"],
            "provincia": row["provincia"],
            "actas_total": data["totalActas"],
            "actas_procesadas": data["contabilizadas"],
            "avance_pct": data["actasContabilizadas"],
            "votos_validos_prov": data["totalVotosValidos"]
        })

    except Exception as e:
        print(f"❌ Error actas {row['provincia']}: {e}")

df_actas = pd.DataFrame(actas)

print("✅ Actas cargadas")

# ================================
# 6. MERGE FINAL
# ================================

df_final = df_votos.merge(
    df_actas,
    on=["departamento", "provincia"]
)

# ================================
# 7. EXPORTAR
# ================================

df_final.to_csv("resultado_provincial_completo.csv", index=False)

print("\n🔥 LISTO")
print("Archivo generado: resultado_provincial_completo.csv")

# ================================
# 8. PREVIEW
# ================================

df_final.head()

Provincias AMAZONAS...
Provincias ANCASH...
Provincias APURIMAC...
Provincias AREQUIPA...
Provincias AYACUCHO...
Provincias CAJAMARCA...
Provincias CALLAO...
Provincias CUSCO...
Provincias HUANCAVELICA...
Provincias HUANUCO...
Provincias ICA...
Provincias JUNIN...
Provincias LA LIBERTAD...
Provincias LAMBAYEQUE...
Provincias LIMA...
Provincias LORETO...
Provincias MADRE DE DIOS...
Provincias MOQUEGUA...
Provincias PASCO...
Provincias PIURA...
Provincias PUNO...
Provincias SAN MARTIN...
Provincias TACNA...
Provincias TUMBES...
Provincias UCAYALI...
✅ Provincias cargadas
Votos BAGUA (AMAZONAS)
Votos BONGARÁ (AMAZONAS)
Votos CHACHAPOYAS (AMAZONAS)
Votos CONDORCANQUI (AMAZONAS)
Votos LUYA (AMAZONAS)
Votos RODRÍGUEZ DE MENDOZA (AMAZONAS)
Votos UTCUBAMBA (AMAZONAS)
Votos AIJA (ANCASH)
Votos ANTONIO RAIMONDI (ANCASH)
Votos ASUNCIÓN (ANCASH)
Votos BOLOGNESI (ANCASH)
Votos CARHUAZ (ANCASH)
Votos CARLOS FERMÍN FITZCARRALD (ANCASH)
Votos CASMA (ANCASH)
Votos CORONGO (ANCASH)
Votos HUARAZ (ANCASH)

,departamento,provincia,candidato,partido,votos,actas_total,actas_procesadas,avance_pct,votos_validos_prov
0,AMAZONAS,BAGUA,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,7988,250,151,60.4,22446
1,AMAZONAS,BAGUA,,VOTOS EN BLANCO,4609,250,151,60.4,22446
2,AMAZONAS,BAGUA,KEIKO SOFIA FUJIMORI HIGUCHI,FUERZA POPULAR,4179,250,151,60.4,22446
3,AMAZONAS,BAGUA,RICARDO PABLO BELMONT CASSINELLI,PARTIDO CÍVICO OBRAS,2286,250,151,60.4,22446
4,AMAZONAS,BAGUA,,VOTOS NULOS,1926,250,151,60.4,22446


In [2]:
# ================================
# 1. LIMPIEZA BÁSICA
# ================================

import numpy as np

df = df_final.copy()

# Evitar divisiones por 0
df["avance_pct"] = df["avance_pct"].replace(0, np.nan)

# ================================
# 2. EXTRAPOLACIÓN PROVINCIAL
# ================================

df["factor_expansion"] = 100 / df["avance_pct"]

df["votos_estimados"] = df["votos"] * df["factor_expansion"]

# ================================
# 3. VOTOS RESTANTES (insight clave)
# ================================

df["votos_restantes"] = df["votos_estimados"] - df["votos"]

# ================================
# 4. AGREGAR A NIVEL NACIONAL
# ================================

df_nacional = (
    df
    .groupby(["candidato", "partido"], as_index=False)
    .agg({
        "votos": "sum",
        "votos_estimados": "sum",
        "votos_restantes": "sum"
    })
)

# ================================
# 5. IDENTIFICAR NO VÁLIDOS
# ================================

no_validos = ["VOTOS EN BLANCO", "VOTOS NULOS"]

# Total votos válidos estimados
total_validos = df_nacional[
    ~df_nacional["partido"].isin(no_validos)
]["votos_estimados"].sum()

# ================================
# 6. % VOTOS VÁLIDOS
# ================================

df_nacional["pct_votos_validos"] = df_nacional.apply(
    lambda row: row["votos_estimados"] / total_validos * 100
    if row["partido"] not in no_validos else None,
    axis=1
)

# ================================
# 7. ORDENAR
# ================================

df_nacional = df_nacional.sort_values(
    by="votos_estimados",
    ascending=False
)

# ================================
# 8. TOP 20
# ================================

top20 = df_nacional.head(20)

print("\n🔥 TOP 20 NACIONAL (EXTRAPOLACIÓN PROVINCIAL)\n")
print(top20)

# ================================
# 9. EXPORTAR
# ================================

top20.to_csv("top20_nacional_provincial.csv", index=False)

print("\n✅ Archivo generado: top20_nacional_provincial.csv")


🔥 TOP 20 NACIONAL (EXTRAPOLACIÓN PROVINCIAL)

                               candidato  \
18          KEIKO SOFIA FUJIMORI HIGUCHI   
0                                          
31      ROBERTO HELBERT SANCHEZ PALOMINO   
27  RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA   
15                JORGE NIETO MONTESINOS   
29      RICARDO PABLO BELMONT CASSINELLI   
8          CARLOS GONSALO ALVAREZ LOAYZA   
24         PABLO ALFONSO LOPEZ CHAU NAVA   
1                                          
20             MARIA SOLEDAD PEREZ TELLO   
3    ALFONSO CARLOS ESPA Y GARCES-ALVEAR   
19            LUIS FERNANDO OLIVERA VEGA   
17                 JOSE LEON LUNA GALVEZ   
37                 YONHY LESCANO ANCIETA   
9                    CESAR ACUÑA PERALTA   
26        PITTER ENRIQUE VALDERRAMA PEÑA   
13         GEORGE PATRICK FORSYTH SOMMER   
21        MARIO ENRIQUE VIZCARRA CORNEJO   
14              HERBERT CALLER GUTIERREZ   
32       RONALD DARWIN ATENCIO SOTOMAYOR   

                            

In [3]:
df_sanchez = df[df["candidato"] == "ROBERTO HELBERT SANCHEZ PALOMINO"]

df_sanchez.sort_values("votos_restantes", ascending=False).head(20)

,departamento,provincia,candidato,partido,votos,actas_total,actas_procesadas,avance_pct,votos_validos_prov,factor_expansion,votos_estimados,votos_restantes
2090,CAJAMARCA,CELENDÍN,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,187,270,2,0.741,260,134.952767,25236.167341,25049.167341
5890,PIURA,HUANCABAMBA,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,2713,356,43,12.079,5136,8.278831,22460.468582,19747.468582
2128,CAJAMARCA,CHOTA,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,21273,465,250,53.763,40442,1.860015,39568.104458,18295.104458
2318,CAJAMARCA,SAN IGNACIO,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,17079,408,203,49.755,29915,2.009848,34326.198372,17247.198372
2052,CAJAMARCA,CAJAMARCA,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,27084,1021,629,61.606,117743,1.623219,43963.250333,16879.250333
5852,PIURA,AYABACA,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,4619,385,86,22.338,8559,4.476677,20677.768824,16058.768824
4144,JUNIN,SATIPO,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,116,545,4,0.734,473,136.239782,15803.814714,15687.814714
6992,SAN MARTIN,TOCACHE,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,242,233,4,1.717,594,58.241118,14094.350612,13852.350612
5102,LIMA,LIMA,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,140340,26368,24031,91.137,5077950,1.097249,153987.952204,13647.952204
2850,CUSCO,LA CONVENCIÓN,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,10576,632,285,45.095,42084,2.217541,23452.710944,12876.710944


In [4]:
df["avance_pct"].describe()

count    7296.000000
mean       69.131328
std        28.645066
min         0.734000
25%        55.653000
50%        80.358500
75%        92.628000
max       100.000000
Name: avance_pct, dtype: float64

In [5]:
df[df["avance_pct"] < 30].groupby("candidato")["votos"].sum().sort_values(ascending=False)

candidato
ROBERTO HELBERT SANCHEZ PALOMINO            33613
                                            31721
KEIKO SOFIA FUJIMORI HIGUCHI                16475
RICARDO PABLO BELMONT CASSINELLI            11481
PABLO ALFONSO LOPEZ CHAU NAVA                5403
CARLOS GONSALO ALVAREZ LOAYZA                4326
JORGE NIETO MONTESINOS                       4017
RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA         3536
JOSE LEON LUNA GALVEZ                        2664
RONALD DARWIN ATENCIO SOTOMAYOR              2236
ALFONSO CARLOS ESPA Y GARCES-ALVEAR          1746
MARIA SOLEDAD PEREZ TELLO                    1485
LUIS FERNANDO OLIVERA VEGA                   1472
YONHY LESCANO ANCIETA                        1372
CESAR ACUÑA PERALTA                          1298
GEORGE PATRICK FORSYTH SOMMER                1283
HERBERT CALLER GUTIERREZ                     1088
MARIO ENRIQUE VIZCARRA CORNEJO               1048
CHARLIE CARRASCO SALAZAR                      992
VLADIMIR ROY CERRON ROJAS               

In [6]:
df_bajo = df[df["avance_pct"] < 30]

df_bajo.groupby("candidato")["votos"].sum().sort_values(ascending=False).head(5)

candidato
ROBERTO HELBERT SANCHEZ PALOMINO    33613
                                    31721
KEIKO SOFIA FUJIMORI HIGUCHI        16475
RICARDO PABLO BELMONT CASSINELLI    11481
PABLO ALFONSO LOPEZ CHAU NAVA        5403
Name: votos, dtype: int64

In [7]:
df_alto = df[df["avance_pct"] > 80]

df_alto.groupby("candidato")["votos"].sum().sort_values(ascending=False).head(5)

candidato
                                        1845903
KEIKO SOFIA FUJIMORI HIGUCHI            1789646
RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA    1565954
JORGE NIETO MONTESINOS                  1438624
RICARDO PABLO BELMONT CASSINELLI        1119108
Name: votos, dtype: int64